In [1]:
# ==============================================================================
# FILE/CELL 1: sensitivity_kmin.py
# Sensitivity of the threshold estimator to the lower truncation bound (k_min)
# ==============================================================================

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
import multiprocessing
from udrud_framework import calculate_k_max, detect_plateau_and_estimate
import matplotlib.pyplot as plt

# 1. Setup Parameters
p = 0.05             # Fixed tail depth
window_size = 5      # Fixed smoothing window
k_min_test_list = [10, 15, 20, 25, 30, 35, 40]

n_cores = multiprocessing.cpu_count()

# 2. Define the parallelized scenario runner
def compute_kmin_scenario(k_min_val, k_max_val, y, w, ws):
    """Runs plateau detection for a specific k_min."""
    k_range_obj = range(k_min_val, max(k_min_val + 1, k_max_val))
    gamma_hat, best_k = detect_plateau_and_estimate(y, w, k_range_obj, window_size=ws)
    return {'k_min': k_min_val, 'gamma_hat': gamma_hat, 'k^*': best_k}

# Load your empirical dataset
df = pd.read_csv("2021_2025_disposable_income.csv")

# equivalize income and weight
df["weight"] = df["weight"] * df["size"]
df["income"] = df["income"] / np.sqrt(df["size"])

# 3. Load Data (assuming df is already loaded and equivalized)
results_kmin = []

for year, group in df.groupby("year"):
    y_val = group["income"].values
    w_val = group["weight"].values

    # Replaced manual calculation with function call
    k_max = calculate_k_max(y_val, w_val, p)

    # Run scenarios in parallel across the k_min list
    rs = Parallel(n_jobs=n_cores)(
        delayed(compute_kmin_scenario)(k_min, k_max, y_val, w_val, window_size)
        for k_min in k_min_test_list
    )

    for res_dict in rs:
        res_dict['year'] = year
        results_kmin.append(res_dict)

# 4. Results
df_kmin_results = pd.DataFrame(results_kmin)


In [2]:
print(df_kmin_results)

    k_min     gamma_hat   k^*  year
0      10 -38180.703019   988  2021
1      15 -38180.703019   988  2021
2      20 -38180.703019   988  2021
3      25 -38180.703019   988  2021
4      30 -38180.703019   988  2021
5      35 -38180.703019   988  2021
6      40 -38180.703019   988  2021
7      10  -5853.680013  1032  2022
8      15  -5853.680013  1032  2022
9      20  -5853.680013  1032  2022
10     25  -5853.680013  1032  2022
11     30  -5853.680013  1032  2022
12     35  -5853.680013  1032  2022
13     40  -5853.680013  1032  2022
14     10 -21090.098056  1039  2023
15     15 -21090.098056  1039  2023
16     20 -21090.098056  1039  2023
17     25 -21090.098056  1039  2023
18     30 -21090.098056  1039  2023
19     35 -21090.098056  1039  2023
20     40 -21090.098056  1039  2023
21     10 -18489.749215  1086  2024
22     15 -18489.749215  1086  2024
23     20 -18489.749215  1086  2024
24     25 -18489.749215  1086  2024
25     30 -18489.749215  1086  2024
26     35 -18489.749215  108